In [1]:
from pypot.dynamixel import DxlIO
from pypot.dynamixel.protocol.v1 import *

from glob import glob

ports = glob('/dev/ttyACM*')
assert len(ports) == 1

port = ports[0]
print('Connecting on port: {}'.format(port))
dxl_io = DxlIO(port)

poulpe_id = 42
N_AXIS = 2

Connecting on port: /dev/ttyACM0


In [17]:
ping_packet = DxlPingPacket(poulpe_id)
dxl_io._send_packet(ping_packet)

DxlStatusPacket(id=42, error=0, parameters=())

In [10]:
import struct

def read_current_pos():
    pos_packet = DxlReadDataPacket(poulpe_id, 50, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_current_pos()

(0.0, 0.0)

In [26]:
import struct

def read_target_position():
    pos_packet = DxlReadDataPacket(poulpe_id, 60, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet,wait_for_status_packet=True)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_target_position()

(0.5, 0.5)

In [25]:
def write_target_position(target):
    p = DxlWriteDataPacket(poulpe_id, 60, struct.pack(N_AXIS * 'f', *target))
    resp = dxl_io._send_packet(p,wait_for_status_packet=True)
    return resp

write_target_position([0.5, 0.5])

DxlStatusPacket(id=42, error=0, parameters=())

In [13]:
def read_torque_enabled():
    p = DxlReadDataPacket(poulpe_id, 40, N_AXIS)
    resp = dxl_io._send_packet(p)
    data = bytearray(resp.parameters)
    torque = struct.unpack(N_AXIS * '?', data)
    return torque

read_torque_enabled()

(True, True)

In [14]:
def write_torque_enabled(torque):
    p = DxlWriteDataPacket(poulpe_id, 40, struct.pack(N_AXIS * '?', *torque))
    resp = dxl_io._send_packet(p)
    return resp

write_torque_enabled([False, False])

DxlStatusPacket(id=42, error=0, parameters=())

In [15]:
write_torque_enabled([True, True])

DxlStatusPacket(id=42, error=0, parameters=())

In [35]:
import time
import numpy as np

pos = []
send_target = []
read_target = []


t0 = time.time()
while True:
    if time.time() - t0 > 5:
        break

    target = [
        10 * np.sin(2 * np.pi * 0.5 * (time.time()-t0)), 
        10 * np.sin(2 * np.pi * 0.5 * (time.time()-t0)),
    ]
    write_target_position(target)
    #time.sleep(0.001)
    send_target.append(target)
    time.sleep(0.01)
    pos.append(read_current_pos())
    read_target.append(read_target_position())
    time.sleep(0.01)

DxlTimeoutError: motors 42 did not respond after sending DxlReadDataPacket(id=42, address=60, length=8)

In [19]:
np.array(send_target) - np.array(read_target)

ValueError: operands could not be broadcast together with shapes (61,2) (60,2) 